Extraction DB

In [0]:
import getpass
import time

In [0]:
path_output = getpass.getpass("Path output to save the tables: ") 
jdbcUrl = getpass.getpass("DB url: ")
connectionProperties = {
    "user": getpass.getpass("DB username: "),
    "password": getpass.getpass("DB password: "),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
def get_all_db_tables():
    query = """SELECT
        TABLE_NAME, 
        TABLE_SCHEMA 
    FROM INFORMATION_SCHEMA.TABLES 
    WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA != 'dbo'"""
    df = spark.read.jdbc(url=jdbcUrl, table=f"({query}) as tables", properties=connectionProperties)
    
    return df

def extract_table(schema, table):
    try:
        df = spark.read.jdbc(url=jdbcUrl, table=f"{schema}.{table}", properties=connectionProperties)
        return df
    
    except Exception as e:
        print(f"Error to extract {schema}.{table}: {e}")
        return None

def save_table(df, schema, table):
    try:
        df.write.format("delta").mode("overwrite").saveAsTable(f'{path_output}.raw_{schema}_{table}')
    except Exception as e:
        print(f"Error to save {schema}.{table}: {e}")
        return None
    
def el_tables_db():
    time_start = time.time()
    tables_df = get_all_db_tables()
    total_tables = tables_df.count()
    count = 1

    print(f'Total tables: {total_tables}')
    for table, schema in tables_df.toLocalIterator():
        print(f'Extracting {count}-{total_tables}: {schema}.{table}')
        df = extract_table(schema, table)
        if df is not None:
            save_table(df, schema, table)
        
        count += 1
    
    time_end = time.time()
    print('Migration complete to delta lake')
    print(f"Total time: {time_end - time_start}")

el_tables_db()